In [1]:
from pathlib import Path
from ultralytics import YOLO

current_dir = Path.cwd().resolve()

if current_dir.name == "notebooks":
    CV_MODEL = current_dir.parent
elif current_dir.name == "cv_model":
    CV_MODEL = current_dir
else:
    CV_MODEL = current_dir / "cv_model"

# Пути к модели, датасету и результатам
MODEL_PATH = CV_MODEL / "models" / "pretrained" / "yolo11n.pt"
DATA_PATH = CV_MODEL / "yolo_person_dataset" / "data.yaml"
RESULTS_PATH = CV_MODEL / "results"

print("Модель:", MODEL_PATH.exists())
print("Датасет:", DATA_PATH.exists())

Модель: True
Датасет: True


In [3]:
# Загружаем исходную YOLO11n
model = YOLO(MODEL_PATH)

# Запускаем дообучение
model.train(
    data=str(DATA_PATH),
    epochs=50,
    patience=10,
    imgsz=640,
    batch=8,
    device="mps",
    workers=0,
    seed=42,
    project=str(RESULTS_PATH),
    name="yolo11n_person_v1",
    exist_ok=True,
)

New https://pypi.org/project/ultralytics/8.4.117 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.115 🚀 Python-3.13.3 torch-2.12.1 MPS (Apple M3)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/kristinaananova/Desktop/SummerPractice2026/cv_model/yolo_person_dataset/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_rat

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x3109f5c50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048,    

In [ ]:
#Проверка на test

# Загружаем лучшие веса
BEST_MODEL = (
    CV_MODEL
    / "models"
    / "yolo11n_person_v1"
    / "best.pt"
)

best_model = YOLO(BEST_MODEL)

# Проверяем на независимом test
metrics = best_model.val(
    data=str(DATA_PATH),
    split="test",
    imgsz=640,
    device="mps",
    classes=[0],
    plots=False,
    project=str(RESULTS_PATH),
    name="yolo11n_person_v1_test",
    exist_ok=True,
)

print("Precision:", round(metrics.box.mp, 4))
print("Recall:", round(metrics.box.mr, 4))
print("mAP50:", round(metrics.box.map50, 4))
print("mAP50-95:", round(metrics.box.map, 4))

Ultralytics 8.4.115 🚀 Python-3.13.3 torch-2.12.1 MPS (Apple M3)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 253.4±105.0 MB/s, size: 69.8 KB)
val: Scanning /Users/kristinaananova/Desktop/SummerPractice2026/cv_model/yolo_person_dataset/labels/test/camera_1.cache... 97 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 120/120 55.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 5.8it/s 1.4s0.2s
                   all        120         97       0.96      0.988      0.975      0.564
Speed: 0.1ms preprocess, 3.2ms inference, 0.0ms loss, 3.4ms postprocess per image
Precision: 0.9599
Recall: 0.9875
mAP50: 0.9747
mAP50-95: 0.564


In [6]:
# Сохраняем предсказания для обеих камер
for camera in ["camera_1", "camera_2"]:
    best_model.predict(
        source=str(
            CV_MODEL
            / "yolo_person_dataset"
            / "images"
            / "test"
            / camera
        ),
        conf=0.2,
        imgsz=640,
        classes=[0],
        device="mps",
        save=True,
        project=str(RESULTS_PATH),
        name=f"predictions_{camera}",
        exist_ok=True,
    )


image 1/43 /Users/kristinaananova/Desktop/SummerPractice2026/cv_model/yolo_person_dataset/images/test/camera_1/frame_000235.jpg: 384x640 (no detections), 12.6ms
image 2/43 /Users/kristinaananova/Desktop/SummerPractice2026/cv_model/yolo_person_dataset/images/test/camera_1/frame_000236.jpg: 384x640 (no detections), 13.6ms
image 3/43 /Users/kristinaananova/Desktop/SummerPractice2026/cv_model/yolo_person_dataset/images/test/camera_1/frame_000237.jpg: 384x640 (no detections), 7.3ms
image 4/43 /Users/kristinaananova/Desktop/SummerPractice2026/cv_model/yolo_person_dataset/images/test/camera_1/frame_000238.jpg: 384x640 (no detections), 13.1ms
image 5/43 /Users/kristinaananova/Desktop/SummerPractice2026/cv_model/yolo_person_dataset/images/test/camera_1/frame_000239.jpg: 384x640 1 person, 9.6ms
image 6/43 /Users/kristinaananova/Desktop/SummerPractice2026/cv_model/yolo_person_dataset/images/test/camera_1/frame_000240.jpg: 384x640 1 person, 13.7ms
image 7/43 /Users/kristinaananova/Desktop/SummerP